<a href="https://colab.research.google.com/github/wlc714/UTC-5440-Assignment_2/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pickle
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pandas as pd
import itertools
import csv
import os
import urllib.request
import tarfile

In [2]:
url = "https://www.cs.toronto.edu/~kriz/cifar-100-python.tar.gz"
urllib.request.urlretrieve(url, "cifar-100-python.tar.gz")

with tarfile.open("cifar-100-python.tar.gz", "r:gz") as tar:
  tar.extractall(".", filter='data')

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [4]:
p = {
    'units': [120, 240],
    'hidden_activations': ['relu', 'sigmoid'],
    'activation': ['softmax', 'sigmoid'],
    'loss': ['mse', 'categorical_crossentropy'],
    'optimizer': ['adam', 'adagrad'],
    'batch_size': [1000, 2000]
}

In [5]:
# with open('./train', 'rb') as f:
#     train_dict = pickle.load(f, encoding='bytes')
# with open('./test', 'rb') as f:
#     test_dict = pickle.load(f, encoding='bytes')

In [6]:
with open('./cifar-100-python/train', 'rb') as f:
    train_dict = pickle.load(f, encoding='bytes')
with open('./cifar-100-python/test', 'rb') as f:
    test_dict = pickle.load(f, encoding='bytes')

In [7]:
X_train = train_dict[b'data'].astype('float32') / 255.0
y_train = np.array(train_dict[b'coarse_labels'])

In [8]:
X_test = test_dict[b'data'].astype('float32') / 255.0
y_test = np.array(test_dict[b'coarse_labels'])

In [9]:
print('x_train shape:', X_train.shape)
print(X_train.shape[0], 'train samples')
print(X_test.shape[0], 'test samples')

x_train shape: (50000, 3072)
50000 train samples
10000 test samples


In [10]:
# Create Coarse Labels
num_classes = 20
print(f'Number of coarse classes: {num_classes}')

Number of coarse classes: 20


In [11]:
# One-hot labels for MSE loss
y_train_oh = np.eye(num_classes, dtype='float32')[y_train]
y_test_oh  = np.eye(num_classes, dtype='float32')[y_test]

In [12]:
def build_model(params):
    # Hidden Activations
    if params['hidden_activations'] == 'relu':
        hidden_activation = nn.ReLU
    else:
        hidden_activation = nn.Sigmoid

    # Output activation
    use_ce = (params['loss'] == 'categorical_crossentropy')

    # Units
    units = params['units']

    # Create Layers Data Object
    layers = []

    # Add first layer - hidden
    layers.append(nn.Linear(3072, units))
    layers.append(hidden_activation())

    # Hidden layers 2-5
    for _ in range(4):
        layers.append(nn.Linear(units, units))
        layers.append(hidden_activation())

    # Create Output Layer
    layers.append(nn.Linear(units, num_classes))

    # Add output activation only if not using CrossEntropyLoss
    if not use_ce:
        if params['activation'] == 'sigmoid':
            layers.append(nn.Sigmoid())
        else:  # softmax
            layers.append(nn.Softmax(dim=1))

    return nn.Sequential(*layers).to(device)

In [13]:
def train_and_evaluate(X_train, y_train_oh, y_train_idx, X_val, y_val_oh, y_val_idx, params):
    use_ce = (params['loss'] == 'categorical_crossentropy')

    model = build_model(params)

    # Setup optimizer
    opt = (optim.Adam(model.parameters())
           if params['optimizer'] == 'adam'
           else optim.Adagrad(model.parameters()))

    criterion = nn.CrossEntropyLoss() if use_ce else nn.MSELoss()

    # CrossEntropyLoss expects integer indices, MSELoss expects one-hot
    if use_ce:
        dataset = TensorDataset(torch.tensor(X_train), torch.tensor(y_train_idx, dtype=torch.long))
    else:
        dataset = TensorDataset(torch.tensor(X_train), torch.tensor(y_train_oh, dtype=torch.float32))

    loader = DataLoader(dataset, batch_size=params['batch_size'], shuffle=True)

    model.train()
    for epoch in range(200):
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            opt.zero_grad()
            output = model(X_batch)
            loss = criterion(output, y_batch)
            loss.backward()
            opt.step()

    model.eval()
    with torch.no_grad():
        train_out = model(torch.tensor(X_train).to(device))
        val_out = model(torch.tensor(X_val).to(device))

        train_acc = (train_out.argmax(dim=1) == torch.tensor(y_train_idx).to(device)).float().mean().item()
        val_acc = (val_out.argmax(dim=1) == torch.tensor(y_val_idx).to(device)).float().mean().item()

    return {'accuracy': round(train_acc, 4), 'val_accuracy': round(val_acc, 4)}

In [14]:
keys = list(p.keys())
combinations = list(itertools.product(*p.values()))
results = []

In [15]:
for i, combo in enumerate(combinations):
    params = dict(zip(keys, combo))
    print(f"[{i+1}/{len(combinations)}] Running: {params}")
    metrics = train_and_evaluate(X_train, y_train_oh, y_train, X_test, y_test_oh, y_test, params)
    results.append({**params, **metrics})
    print(f"  => acc={metrics['accuracy']:.4f}  val_acc={metrics['val_accuracy']:.4f}")

    # Write after every permutation
    os.makedirs('grid_search_output', exist_ok=True)
    with open('grid_search_output/results.csv', 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(results[0].keys()))
        writer.writeheader()
        writer.writerows(results)

[1/64] Running: {'units': 120, 'hidden_activations': 'relu', 'activation': 'softmax', 'loss': 'mse', 'optimizer': 'adam', 'batch_size': 1000}
  => acc=0.6885  val_acc=0.3414
[2/64] Running: {'units': 120, 'hidden_activations': 'relu', 'activation': 'softmax', 'loss': 'mse', 'optimizer': 'adam', 'batch_size': 2000}
  => acc=0.5791  val_acc=0.3524
[3/64] Running: {'units': 120, 'hidden_activations': 'relu', 'activation': 'softmax', 'loss': 'mse', 'optimizer': 'adagrad', 'batch_size': 1000}
  => acc=0.4473  val_acc=0.3423
[4/64] Running: {'units': 120, 'hidden_activations': 'relu', 'activation': 'softmax', 'loss': 'mse', 'optimizer': 'adagrad', 'batch_size': 2000}
  => acc=0.4196  val_acc=0.3390
[5/64] Running: {'units': 120, 'hidden_activations': 'relu', 'activation': 'softmax', 'loss': 'categorical_crossentropy', 'optimizer': 'adam', 'batch_size': 1000}
  => acc=0.6442  val_acc=0.3339
[6/64] Running: {'units': 120, 'hidden_activations': 'relu', 'activation': 'softmax', 'loss': 'categori

In [16]:
df = pd.read_csv('grid_search_output/results.csv')
print(df.sort_values('val_accuracy', ascending=False).to_string(index=False))

 units hidden_activations activation                     loss optimizer  batch_size  accuracy  val_accuracy
   240               relu    sigmoid categorical_crossentropy   adagrad        2000    0.4924        0.3579
   240               relu    sigmoid categorical_crossentropy   adagrad        1000    0.6009        0.3573
   240               relu    softmax categorical_crossentropy   adagrad        1000    0.5477        0.3569
   120               relu    softmax categorical_crossentropy      adam        2000    0.5668        0.3568
   120               relu    sigmoid categorical_crossentropy      adam        2000    0.5735        0.3553
   120               relu    softmax                      mse      adam        2000    0.5791        0.3524
   120               relu    sigmoid                      mse   adagrad        1000    0.4580        0.3508
   240               relu    softmax categorical_crossentropy   adagrad        2000    0.4785        0.3507
   120               relu   